In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

In [47]:
PROCESSED_PATH = Path("../data/processed")
EMBEDDINGS_PATH = Path("../data/embeddings")
MODELS_PATH = Path("../models")

MODELS_PATH.mkdir(parents=True, exist_ok=True)

## Wczytanie danych

In [48]:
df = pd.read_parquet(PROCESSED_PATH / "movies_processed.parquet")
text_embeddings = np.load(EMBEDDINGS_PATH / "text_embeddings.npy")
genres_embeddings = np.load(EMBEDDINGS_PATH / "genres_embeddings.npy")
movie_ids = np.load(EMBEDDINGS_PATH / "movie_ids.npy")
genre_names = np.load(EMBEDDINGS_PATH / "genre_names.npy")

## Normalizacja reprezentacji gatunków i połączenie reprezentacji

In [49]:
genres_normalized = normalize(genres_embeddings, norm="l2")

In [50]:
def combined_features(text_embeddings, genres_embeddings, text_weight=1.0, genres_weight=0.0):
    text_part = text_embeddings * text_weight
    genres_part = genres_embeddings * genres_weight

    return np.hstack([
        text_part, genres_part
    ]).astype("float32")

### K-Means
1. Profilowanie użytkownika (profil preferencji, jakie grupy filmów użytkownik ocenia najwyżej)
2. Premiowanie filmów z preferowanych klastrów

In [51]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = combined_features(text_embeddings, genres_normalized, text_weight=1.0,genres_weight=1.0)

k_values = [5, 10, 15, 20, 30]
results = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = kmeans.fit_predict(X)
    silhouette = silhouette_score(X,labels, sample_size=5000, random_state=42)

    results.append({
        "k": k,
        "silhouette": silhouette,
        "inertia": kmeans.inertia_
    })

results_df = pd.DataFrame(results)

display(results_df)

,k,silhouette,inertia
0,5,0.116699,53395.320312
1,10,0.094571,47843.093750
2,15,0.095795,45151.437500
3,20,0.095826,43274.441406
4,30,0.067215,41431.820312


In [52]:

X_best = combined_features(text_embeddings, genres_normalized, text_weight=1.0, genres_weight=1.0)
kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)

df["cluster"] = kmeans.fit_predict(X_best)

In [53]:
joblib.dump(kmeans, MODELS_PATH / "kmeans.pkl")
df[["id", "cluster"]].to_parquet(PROCESSED_PATH / "movie_clusters.parquet", index=False)